# 00 · Configuración de la base de datos

Crea `data.db` y su esquema. Se ejecuta una sola vez al inicio del proyecto,
o cuando haga falta reconstruir alguna tabla desde cero.

| Tabla | Contenido | Columnas |
|---|---|---|
| `metadata` | Catálogo WikiArt: artista, género, movimiento | 4 |
| `mfdfa_b1` | MF-DFA, banda 6 px a 25% | 197 |
| `mfdfa_b2` | MF-DFA, banda 25% a 75% | 197 |
| `mfrenyi_b1` | MF-Rényi, banda 6 px a 25% | 183 |
| `mfrenyi_b2` | MF-Rényi, banda 25% a 75% | 183 |

Todas se ligan por `painting_id`, que es la posición de la obra en el dataset.

**La extracción de características no ocurre aquí.** Este notebook solo define
el contenedor; los notebooks 01 en adelante lo llenan.

In [9]:
import sqlite3
import pandas as pd

import db

con = db.connect()

## Catálogo de obras

Se piden solo las columnas de etiqueta, en modo streaming, así que las
imágenes nunca se descargan.

In [2]:
db.create_metadata_table(con)

Resolving data files:   0%|          | 0/72 [00:00<?, ?it/s]

Table metadata created with 81444 paintings


## Tablas de características

Una tabla por método y por banda. `drop=True` reconstruye desde cero:
úsalo solo cuando quieras perder lo que haya dentro.

In [ ]:
db.create_mfdfa_b1(con, drop=True)
db.create_mfdfa_b2(con, drop=True)
db.create_mfrenyi_b1(con, drop=True)
db.create_mfrenyi_b2(con, drop=True)

## Verificación del esquema

Qué tablas existen, cuántas columnas tiene cada una y cuántas filas lleva.

In [3]:
tablas = [f[0] for f in con.execute(
    "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name"
)]

for t in tablas:
    n_cols = len(list(con.execute(f'PRAGMA table_info("{t}")')))
    n_filas = con.execute(f'SELECT COUNT(*) FROM "{t}"').fetchone()[0]
    print(f"{t:<14} {n_cols:>4} columnas   {n_filas:>7} filas")

metadata          4 columnas     81444 filas
mfdfa_b1        197 columnas         0 filas
mfdfa_b2        197 columnas         0 filas
mfrenyi_b1      183 columnas         0 filas
mfrenyi_b2      183 columnas         0 filas


## Panorama del catálogo

Distribución de las etiquetas antes de aplicar cualquier filtro.

In [10]:
pd.read_sql_query("""
    SELECT movement, COUNT(*) AS obras
    FROM metadata
    GROUP BY movement
    ORDER BY obras DESC
""", con)

,movement,obras
0,Impressionism,13060
1,Realism,10733
2,Romanticism,7019
3,Expressionism,6736
4,Post_Impressionism,6450
5,Symbolism,4528
6,Art_Nouveau,4334
7,Baroque,4240
8,Abstract_Expressionism,2782
9,Northern_Renaissance,2552


In [11]:
pd.read_sql_query("""
    SELECT genre, COUNT(*) AS obras
    FROM metadata
    GROUP BY genre
    ORDER BY obras DESC
""", con)

,genre,obras
0,Unknown Genre,16452
1,portrait,14112
2,landscape,13358
3,genre_painting,10859
4,religious_painting,6538
5,abstract_painting,4968
6,cityscape,4602
7,sketch_and_study,3942
8,still_life,2788
9,nude_painting,1923


In [5]:
pd.read_sql_query("""
    SELECT
        COUNT(*)                          AS total,
        COUNT(DISTINCT artist)            AS artistas,
        COUNT(DISTINCT movement)          AS movimientos,
        COUNT(DISTINCT genre)             AS generos
    FROM metadata
""", con)

,total,artistas,movimientos,generos
0,81444,129,27,11


In [12]:
con.close()